# sinav-okur · TrOCR Türkçe ince ayarı (v3 — start=2 düzeltmesi)

v2 başarısızlığının kök nedeni, checkpointin üretimde beklediği `decoder_start_token_id=2` değerinin
`0` ile değiştirilmesiydi. Kayıp düşse de üretim bir konum kayıyor ve CER bozuluyordu. Bu sürüm
checkpoint başlangıcını korur, kaybı açıkça hizalı hesaplar, benzersiz metin üretir ve metin-ayrık böler.

> Ölçüm sentetik el-yazısı fontları içindir; gerçek sınav/tıbbi not doğruluğu ancak izinli,
> yazar-ayrık ve satır transkripsiyonlu gerçek veriyle ayrıca ölçülür.


In [ ]:
import base64, io, sys, zipfile
PKG_B64 = (
    "..."
)
zipfile.ZipFile(io.BytesIO(base64.b64decode(''.join(PKG_B64)))).extractall('/content/pkg')
sys.path.insert(0, '/content/pkg')
import sinav_okur; print('sinav_okur', sinav_okur.__version__)

In [ ]:
!pip install -q 'transformers>=4.40,<5' datasets jiwer evaluate accelerate
import torch; print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

In [ ]:
import os, urllib.request
from sinav_okur.veri import ONERILEN_FONTLAR, fontlari_dogrula
FONT_URL = 'https://raw.githubusercontent.com/google/fonts/main/ofl/{slug}/{dosya}'
DOSYALAR = {'Caveat':'Caveat[wght].ttf','Caveat Brush':'CaveatBrush-Regular.ttf',
  'Dancing Script':'DancingScript[wght].ttf','Pacifico':'Pacifico-Regular.ttf',
  'Patrick Hand':'PatrickHand-Regular.ttf','Kalam':'Kalam-Regular.ttf',
  'Amatic SC':'AmaticSC-Regular.ttf','Indie Flower':'IndieFlower-Regular.ttf',
  'Marck Script':'MarckScript-Regular.ttf','Courgette':'Courgette-Regular.ttf',
  'Great Vibes':'GreatVibes-Regular.ttf','Sacramento':'Sacramento-Regular.ttf',
  'Bad Script':'BadScript-Regular.ttf','Merienda':'Merienda[wght].ttf','Itim':'Itim-Regular.ttf'}
os.makedirs('fontlar', exist_ok=True)
for ad in ONERILEN_FONTLAR:
    try: urllib.request.urlretrieve(FONT_URL.format(slug=ad.lower().replace(' ',''), dosya=DOSYALAR[ad]), f'fontlar/{DOSYALAR[ad]}')
    except Exception as e: print('x', ad, e)
print(sum(1 for e in fontlari_dogrula('fontlar').values() if not e), 'font Türkçe-tam')

## Benzersiz metinli veri (ezber kısayolu kapalı)

In [ ]:
from sinav_okur.veri import veri_uret
rapor = veri_uret('veri', 'fontlar', adet=40000, tohum=11, benzersiz_metin=True)
print(rapor)
assert rapor['benzersiz_metin'] == rapor['uretilen'], 'yinelenen metin kaldı'

In [ ]:
import json
from sinav_okur.veri import metin_ayrik_bol
kayitlar = [json.loads(l) for l in open('veri/labels.jsonl')]
egitim_k, test_k = metin_ayrik_bol(kayitlar, test_orani=0.03, tohum=5)
ortak = {k['text'] for k in egitim_k} & {k['text'] for k in test_k}
print('egitim', len(egitim_k), 'test', len(test_k), 'SIZINTI:', len(ortak))
assert not ortak

In [ ]:
import os
from datasets import Dataset, Image as HfImage
def hazirla(ks):
    ks = [{**k, 'image': os.path.join('veri', k['image'])} for k in ks]
    return Dataset.from_list(ks).cast_column('image', HfImage())
ds_tr, ds_te = hazirla(egitim_k), hazirla(test_k)
print(ds_tr, ds_te)

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel
import torch
AD = 'microsoft/trocr-base-handwritten'
islemci = TrOCRProcessor.from_pretrained(AD)
model = VisionEncoderDecoderModel.from_pretrained(AD)
tk = islemci.tokenizer
START_ID = int(model.generation_config.decoder_start_token_id)
assert START_ID == 2, f'Checkpoint start kimliği değişmiş: {START_ID}'
model.config.decoder_start_token_id = START_ID
model.config.pad_token_id = tk.pad_token_id
model.config.eos_token_id = tk.eos_token_id
model.config.vocab_size = model.config.decoder.vocab_size
model.config.use_cache = False
model.generation_config.decoder_start_token_id = START_ID
model.generation_config.pad_token_id = tk.pad_token_id
model.generation_config.eos_token_id = tk.eos_token_id
model.generation_config.max_new_tokens = 96
model.generation_config.num_beams = 1
print('checkpoint start', START_ID, '| cls/bos/eos/pad', tk.cls_token_id, tk.bos_token_id, tk.eos_token_id, tk.pad_token_id)

In [ ]:
from torch.utils.data import DataLoader

def collate(batch):
    px = islemci(images=[x['image'].convert('RGB') for x in batch], return_tensors='pt').pixel_values
    enc = tk([x['text'] for x in batch], padding=True, truncation=True, max_length=96, return_tensors='pt')
    labels = enc.input_ids.clone()
    labels[labels == tk.pad_token_id] = -100
    return px, labels, [x['text'] for x in batch]

print('Açık hizalı veri toplayıcı hazır; <s> etiketi korunuyor, decoder girişi start=2 ile kaydırılacak.')

In [ ]:
from jiwer import cer as jiwer_cer
print('CER ölçümü hazır')

In [ ]:
import os, json, shutil, torch.nn.functional as F
from transformers import get_cosine_schedule_with_warmup

model = model.cuda()
train_loader = DataLoader(ds_tr, batch_size=32, shuffle=True, num_workers=2, collate_fn=collate, pin_memory=True, persistent_workers=True)
EPOCHS = 4
opt = torch.optim.AdamW(model.parameters(), lr=4e-5, weight_decay=0.01)
sched = get_cosine_schedule_with_warmup(opt, 400, EPOCHS * len(train_loader))

@torch.no_grad()
def degerlendir(ds):
    model.eval(); preds=[]; refs=[]
    for px, _, texts in DataLoader(ds, batch_size=32, num_workers=2, collate_fn=collate, pin_memory=True):
        out = model.generate(px.cuda(non_blocking=True), max_new_tokens=96, num_beams=1, do_sample=False)
        preds += tk.batch_decode(out, skip_special_tokens=True); refs += texts
    return jiwer_cer(refs, preds), list(zip(preds[:5], refs[:5]))

best_cer = float('inf'); history=[]
for epoch in range(1, EPOCHS + 1):
    model.train(); total=0.0
    for step, (px, labels, _) in enumerate(train_loader, 1):
        px=px.cuda(non_blocking=True); labels=labels.cuda(non_blocking=True)
        dec=torch.full_like(labels, tk.pad_token_id); dec[:,0]=START_ID; dec[:,1:]=labels[:,:-1]
        dec.masked_fill_(dec == -100, tk.pad_token_id)
        opt.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=torch.bfloat16):
            logits=model(pixel_values=px, decoder_input_ids=dec, use_cache=False).logits
            loss=F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1), ignore_index=-100)
        loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
        total += float(loss.detach())
        if step % 200 == 0: print(f'E{epoch} {step}/{len(train_loader)} loss={total/step:.4f}')
    val_cer, samples = degerlendir(ds_te)
    row={'epoch':epoch,'train_loss':total/len(train_loader),'val_cer':val_cer,'samples':samples}
    history.append(row); print('EPOCH_SONU', json.dumps(row, ensure_ascii=False))
    if val_cer < best_cer:
        best_cer=val_cer; model.save_pretrained('trocr-tr'); islemci.save_pretrained('trocr-tr')
        with open('trocr-tr/egitim_raporu.json','w') as f:
            json.dump({'best_cer':best_cer,'decoder_start_token_id':START_ID,'history':history}, f, ensure_ascii=False, indent=2)
        print('YENI_EN_IYI', epoch, best_cer)
print('EGITIM_TAMAM', best_cer)

In [ ]:
model = VisionEncoderDecoderModel.from_pretrained('trocr-tr').cuda().eval()
for i in range(5):
    px = islemci(images=ds_te[i]['image'].convert('RGB'), return_tensors='pt').pixel_values.cuda()
    out = model.generate(px, max_new_tokens=96, num_beams=1)
    print(repr(tk.decode(out[0], skip_special_tokens=True)), '<->', repr(ds_te[i]['text']))

In [ ]:
!zip -qr trocr-tr.zip trocr-tr -x 'trocr-tr/checkpoint-*'
import os
print('zip MB:', os.path.getsize('trocr-tr.zip')//1048576)
print('Büyük dosyayı Colab Dosyalar panelinden veya Drive üzerinden kopyalayın; files.download kullanmayın.')